In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 


In [23]:
# dataset using 
deliveris=pd.read_csv('deliveries.csv') # dataset of all the ball of the ipl 
matches=pd.read_csv('matches.csv')      # dataset of the matches record in the ipl 
matches.columns

Index(['id', 'Season', 'city', 'date', 'team1', 'team2', 'toss_winner',
       'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs',
       'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2',
       'umpire3'],
      dtype='str')

# this project is to make a ml model which predict the win-rate of the teams in ipl match 

solution - to predict wheter a team will win or not we have to get the target they are chasing their current run rate , their required run rate , wickets in-hand , batsmen on field , teams that are playing 

In [54]:
df=deliveris.groupby(['match_id','inning']).sum()['total_runs'].reset_index()
df=df[df['inning']==1]
df=df.merge(matches,left_on='match_id',right_on='id')
df.columns

Index(['match_id', 'inning', 'total_runs', 'id', 'Season', 'city', 'date',
       'team1', 'team2', 'toss_winner', 'toss_decision', 'result',
       'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets',
       'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3'],
      dtype='str')

In [59]:
#df.drop(columns=['umpire1','umpire2','umpire3'],inplace=True)
df_c=df.copy()

In [70]:
# renaming old teams name into current 
df_c['team1']=df['team1'].replace({'Delhi Daredevils':'Delhi Capitals', 'Deccan Chargers':'Sunrisers Hyderabad',})
df_c['team2']=df['team2'].replace({'Delhi Daredevils':'Delhi Capitals', 'Deccan Chargers':'Sunrisers Hyderabad',})

In [86]:
# removing teams which now not exist 
team=['Sunrisers Hyderabad',           
'Mumbai Indians',                
'Kings XI Punjab',                
'Chennai Super Kings',          
'Royal Challengers Bangalore',   
'Kolkata Knight Riders',        
'Delhi Capitals',             
'Rajasthan Royals',]
# df_c=df_c[df_c['team1'].isin(team)]
# df_c=df_c[df_c['team2'].isin(team)]
#<--------------------------------------->

# we do not want the data which have rain or dl applied
#df_c=df_c[df_c['dl_applied']==0]

df_c=df_c[['match_id', 'total_runs','winner','city']]

In [97]:
# merging with deliveris 
df_final=deliveris.merge(df_c,on='match_id')
df_f=df_final.copy()


In [ ]:
# runs left to chase
df_f['current_score']=df_f[df_f['inning']==2].groupby('match_id')['total_runs_x'].cumsum()
df_f['run_left']=df_f['total_runs_y']+1-df_f['current_score']


In [135]:
# balls left 
df_f['ball_left']=126-(df_f['over']*6+df_f['ball'])


In [136]:
# wickets_left
# df_f['player_dismissed']=df_f['player_dismissed'].fillna('0')
# df_f['player_dismissed']=df_f['player_dismissed'].apply(lambda x: x if x=='0' else '1')
df_f['player_dismissed']=df_f['player_dismissed'].astype('int')
df_f['wickets_left']=10-df_f[df_f['inning']==2].groupby('match_id')['player_dismissed'].cumsum()


In [151]:
# current run rate 
df_f['crr']=df_f['current_score']*6/(120-df_f['ball_left'])

# required run rate

df_f['rrr']=(df_f['run_left']*6)/df_f['ball_left']

In [ ]:
# adding result

def check(data):
    return 1 if data['winner']==data['batting_team'] else 0

df_f['result']=df_f.apply(check,axis=1)


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,winner,city,current_score,run_left,ball_left,wickets_left,crr,rrr,resultd,result
5835,48,1,Kings XI Punjab,Kolkata Knight Riders,1,3,MJ Guptill,M Vohra,UT Yadav,0,...,Kings XI Punjab,Chandigarh,NaN,NaN,117,NaN,NaN,NaN,1,1
11325,78,1,Rajasthan Royals,Kolkata Knight Riders,4,4,M Kaif,SA Asnodkar,I Sharma,0,...,Rajasthan Royals,Jaipur,NaN,NaN,98,NaN,NaN,NaN,1,1
135081,7953,1,Sunrisers Hyderabad,Chennai Super Kings,12,3,KS Williamson,Shakib Al Hasan,DJ Bravo,0,...,Chennai Super Kings,Mumbai,NaN,NaN,51,NaN,NaN,NaN,0,0
1262,11,1,Kings XI Punjab,Kolkata Knight Riders,10,2,GJ Maxwell,HM Amla,PP Chawla,0,...,Kolkata Knight Riders,Kolkata,NaN,NaN,64,NaN,NaN,NaN,0,0
142921,11323,2,Rajasthan Royals,Kings XI Punjab,3,5,JC Buttler,RA Tripathi,A Singh,0,...,Kings XI Punjab,Mohali,28.0,163.0,103,10.0,9.882353,9.495146,0,0
91529,478,2,Kolkata Knight Riders,Chennai Super Kings,14,1,RV Uthappa,YK Pathan,RA Jadeja,0,...,Chennai Super Kings,Ranchi,85.0,64.0,41,5.0,6.455696,9.365854,0,0
140936,11315,1,Kings XI Punjab,Mumbai Indians,17,5,KK Nair,KL Rahul,HH Pandya,0,...,Mumbai Indians,Mumbai,NaN,NaN,19,NaN,NaN,NaN,0,0
111274,563,1,Chennai Super Kings,Rajasthan Royals,17,4,BB McCullum,P Negi,SR Watson,0,...,Chennai Super Kings,Chennai,NaN,NaN,20,NaN,NaN,NaN,1,1
40500,204,2,Deccan Chargers,Kolkata Knight Riders,1,3,HH Gibbs,AC Gilchrist,MB Parmar,0,...,Kolkata Knight Riders,Kolkata,5.0,177.0,117,10.0,10.000000,9.076923,0,0
15222,94,2,Kings XI Punjab,Royal Challengers Bangalore,9,6,SE Marsh,LA Pomersbach,R Vinay Kumar,0,...,Kings XI Punjab,Chandigarh,66.0,78.0,66,9.0,7.333333,7.090909,1,1
